In [4]:
import csv

path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\UniSuperOptionHoldings (1).csv"

# --- Load all rows ---
with open(path, "r", encoding="cp1252", newline="") as f:
    rows = list(csv.reader(f))

# --- Step 1: isolate the table where "INVESTMENT OPTION NAME" is followed by "Balanced" in the next column ---
# Find all marker rows and the column index of the marker cell
markers = []
for i, row in enumerate(rows):
    for j, c in enumerate(row):
        if str(c).strip().upper() == "INVESTMENT OPTION NAME":
            markers.append((i, j))
            break  # assume one marker cell per row

# Find the Balanced table start/end
start_idx = end_idx = None
for k, (i, j) in enumerate(markers):
    next_val = rows[i][j+1].strip() if j + 1 < len(rows[i]) else ""
    if next_val == "Balanced":  # exact match as requested
        start_idx = i
        end_idx = markers[k + 1][0] if k + 1 < len(markers) else len(rows)
        break

if start_idx is None:
    raise ValueError('Could not find a row where "INVESTMENT OPTION NAME" is immediately followed by "Balanced".')

balanced_rows = rows[start_idx:end_idx]

# --- Step 2: within the Balanced table, keep only TABLE 1 .. (exclude) TOTAL INVESTMENT ITEMS (both in first column) ---
t1_start = t1_end = None
for i, row in enumerate(balanced_rows):
    first = str(row[0]).strip().upper() if row else ""
    if first == "TABLE 1":
        t1_start = i
    elif first == "TOTAL INVESTMENT ITEMS" and t1_start is not None:
        t1_end = i
        break

if t1_start is None or t1_end is None:
    raise ValueError("Could not find 'TABLE 1' start and/or 'TOTAL INVESTMENT ITEMS' end (in first column).")

# Include TABLE 1 row, exclude TOTAL INVESTMENT ITEMS row
selected = balanced_rows[t1_start+3:t1_end]

# --- Overwrite the same file with the selected rows ---
with open(path, "w", encoding="cp1252", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(selected)

print(f"✅ Wrote TABLE 1 from the Balanced section back to the same file. Rows kept: {t1_start}..{t1_end-1} within the Balanced block.")


✅ Wrote TABLE 1 from the Balanced section back to the same file. Rows kept: 4..3602 within the Balanced block.


In [1]:
import csv
import re
import pandas as pd

# Paths
table1_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\UniSuperOptionHoldings (1).csv"
codes_path  = r"D:\LinhDao\Programming\SUPERFUNdProject\InternationalCountryCodes.csv"
out_path    = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned.csv"

# Final schema / headers
final_cols = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]

def is_header_row(row):
    lo = [str(c or "").lower() for c in row]
    return any("name" in c for c in lo) and any("value" in c for c in lo) and any("weight" in c for c in lo)

def is_total_row(row):
    # Strict: first cell must be exactly "TOTAL" (case-sensitive)
    return bool(row) and str(row[0]).strip() == "TOTAL"

def parse_int_ext(text):
    s = str(text or "").strip().lower()
    if "externally managed" in s: return 1
    if "internally managed" in s: return 0
    return ""

def is_asset_class_row(row):
    if not row: return False
    first = str(row[0]).strip()
    if not first: return False
    if first.upper() in {"TABLE 1", "TOTAL", "TOTAL INVESTMENT ITEMS", "INVESTMENT OPTION NAME"}:
        return False
    if any(str(c).strip() for c in row[1:]):  # per your format
        return False
    tokens = re.findall(r"[A-Za-z]+", first)
    return 1 <= len(tokens) <= 2

def map_header(row):
    header_map = {}
    for j, raw in enumerate(row):
        col = str(raw or "").strip()
        if not col: continue
        low = col.lower()
        if low.startswith("name"):
            header_map[j] = "Name/Kind of Investment Item"
        elif "security identifier" in low:
            header_map[j] = "Stock ID"
        elif "% of property held" in low or "% ownership" in low:
            header_map[j] = "% Ownership"
        elif "units held" in low:
            header_map[j] = "Units Held"
        elif low == "currency":
            header_map[j] = "Currency"
        elif low in {"address", "location", "listed country"}:
            header_map[j] = "Address" if low == "address" else "Listed Country"
        elif "value" in low:
            header_map[j] = "Value (AUD)"
        elif "weighting" in low:
            header_map[j] = "Weighting"
    return header_map

# --- Load TABLE 1 csv ---
with open(table1_path, "r", encoding="cp1252", newline="") as f:
    rows = list(csv.reader(f))

all_out = []
i, n = 0, len(rows)

while i < n:
    row = rows[i]
    if is_asset_class_row(row):
        # Title-case the asset class name
        asset_class_name = str(row[0]).strip().title()
        int_ext = ""

        # Find header row, capturing any Int/Ext line before it
        k = i + 1
        header_idx = None
        while k < n:
            text_line = " ".join(str(c).strip() for c in rows[k] if str(c).strip())
            if int_ext == "":
                ie = parse_int_ext(text_line)
                if ie != "":
                    int_ext = ie
            if is_header_row(rows[k]):
                header_idx = k
                break
            if is_total_row(rows[k]):
                break
            k += 1

        if header_idx is None:
            i += 1
            continue

        header_map = map_header(rows[header_idx])

        # Collect rows up to and including TOTAL
        j = header_idx + 1
        while j < n:
            r = rows[j]
            out = {c: "" for c in final_cols}
            out.update({
                "Effective Date": "31/12/2024",
                "Fund Name": "UniSuper",
                "Option Name": "Balanced",
                "Asset Class Name": asset_class_name,
                "Int/Ext": int_ext,
            })
            for idx, tgt in header_map.items():
                if idx < len(r):
                    out[tgt] = str(r[idx]).strip()
            all_out.append(out)

            if is_total_row(r):
                j += 1
                break
            j += 1

        i = j
        continue
    i += 1

# Build DataFrame
df_out = pd.DataFrame(all_out, columns=final_cols)

# --- Post-processing ---
# 1) Int/Ext: fill blanks with 1 (int)
df_out["Int/Ext"] = df_out["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)

# 2) Stock ID -> Listed Country (first 2 chars), and trim Stock ID
def split_stockid(val):
    if pd.isna(val) or not str(val).strip():
        return ("", "")
    s = str(val).strip()
    if len(s) > 2:
        return (s[2:], s[:2])
    return (s, "")
df_out["Stock ID"], df_out["Listed Country"] = zip(*df_out["Stock ID"].map(split_stockid))

# 3) Weighting: remove % and divide by 100
def clean_weight(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    s = str(val).strip().replace("%", "")
    try:
        return float(s) / 100.0
    except:
        return s
df_out["Weighting"] = df_out["Weighting"].map(clean_weight)

# 4) Convert literal "TOTAL" names to "Sub Total"
df_out.loc[df_out["Name/Kind of Investment Item"].str.strip().eq("TOTAL"), "Name/Kind of Investment Item"] = "Sub Total"

# 5) Country code lookup (Code -> Country) with BOM-safe read
try:
    lu = pd.read_csv(codes_path, encoding="utf-8-sig")
    lu.columns = [str(c).replace("\ufeff", "").strip() for c in lu.columns]  # normalize headers
    code_to_country = dict(
        zip(
            lu["Code"].astype(str).str.strip().str.upper(),
            lu["Country"].astype(str).str.strip(),
        )
    )

    def lookup_country(code):
        if pd.isna(code) or str(code).strip() == "":
            return ""
        s = str(code).strip().upper()
        return code_to_country.get(s, s)

    df_out["Listed Country"] = df_out["Listed Country"].map(lookup_country)
except Exception as e:
    print(f"⚠️ Country lookup skipped due to error: {e}")

# Save
df_out.to_csv(out_path, index=False, encoding="cp1252")
print(f"✅ Cleaned file saved to: {out_path} — rows: {len(df_out)}")


✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned.csv — rows: 3541


C:\Users\thuon\AppData\Local\Temp\ipykernel_18084\1886206531.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["Int/Ext"] = df_out["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)
